In [2]:
pip install einops

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 KB 4.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [2]:
import torch
from use_croma import PretrainedCROMA

# 1. Initialize the model
model = PretrainedCROMA(
    pretrained_path=r"/home/arm/Desktop/ARM/Codes/CROMA/CROMA_base.pt",
    size='base',
    modality='SAR',
    image_resolution=224
)

# 2. Print the full architecture
print("\n--- Full Model Architecture ---")
print(model)

# 3. Print specifically the SAR Encoder (where we attached hooks)
print("\n--- SAR Encoder (s1_encoder) ---")
print(model.s1_encoder)

Initializing SAR encoder

--- Full Model Architecture ---
PretrainedCROMA(
  (s1_encoder): ViT(
    (linear_input): Linear(in_features=128, out_features=768, bias=True)
    (transformer): BaseTransformer(
      (layers): ModuleList(
        (0-5): 6 x ModuleList(
          (0): Attention(
            (to_qkv): Linear(in_features=768, out_features=2304, bias=False)
            (to_out): Linear(in_features=768, out_features=768, bias=True)
            (input_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (1): FFN(
            (net): Sequential(
              (0): Linear(in_features=768, out_features=3072, bias=True)
              (1): GELU(approximate='none')
              (2): Dropout(p=0.0, inplace=False)
              (3): Linear(in_features=3072, out_features=768, bias=True)
            )
            (input_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          )
        )
      )

huge model. patch size 8x8 (784 patches). need small batch size.

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm
import rasterio
from pathlib import Path
import kornia.augmentation as K
import kornia.augmentation.container as C
from torch.amp import autocast, GradScaler

# SMP Imports
from segmentation_models_pytorch.decoders.upernet.decoder import UPerNetDecoder
from segmentation_models_pytorch.base import SegmentationHead

# --- CHANGED: Import CROMA dependencies ---
# Assuming you have the CROMA classes (PretrainedCROMA, ViT, etc.) 
# saved in a file named 'use_croma.py' or pasted above this script.
# If they are in the same file, we just use the classes directly.
from use_croma import PretrainedCROMA 

# ============================================================================
# 1. Feature Extraction Helper (MODIFIED for CROMA)
# ============================================================================

class CromaFeatureExtractor:
    def __init__(self, croma_model):
        # CROMA's SAR encoder is stored in 's1_encoder'
        self.model = croma_model.s1_encoder
        self.features = {}
        self.hooks = []
        
        # CROMA SAR Base encoder usually has 6 layers (half of 12).
        # We hook early, mid, and late layers for the Pyramid.
        # Layers are in: model.s1_encoder.transformer.layers
        
        # Hooking Layer 1 (Stage 1)
        self.hooks.append(self.model.transformer.layers[0][1].register_forward_hook(self._get_hook('block_0')))
        # Hooking Layer 3 (Stage 3)
        self.hooks.append(self.model.transformer.layers[2][1].register_forward_hook(self._get_hook('block_2')))
        # Hooking Layer 5 (Stage 5)
        self.hooks.append(self.model.transformer.layers[4][1].register_forward_hook(self._get_hook('block_4')))
        
        # Hook the FINAL NORM layer
        self.hooks.append(self.model.transformer.norm_out.register_forward_hook(self._get_hook('norm')))

    def _get_hook(self, name):
        def hook(model, input, output):
            self.features[name] = output
        return hook

    def clear(self):
        self.features = {}

def extract_patch_features(model, images, feature_extractor):
    feature_extractor.clear()
    

    # We must access the internal s1_encoder to run forward
    # because the hooks are attached to it.
    attn_bias = model.attn_bias.to(images.device)
    
    # Run forward. We ignore output, we just want hooks to fire.
    _ = model.s1_encoder(images, attn_bias=attn_bias)
    
    return feature_extractor.features

# ============================================================================
# 2. Decoder (MODIFIED for CROMA)
# ============================================================================

class CromaUPerNet(nn.Module):
    def __init__(self, encoder_dim=768, decoder_channels=256, num_classes=2, patch_size=8, dropout=0.1):
        super().__init__()
        # CROMA uses patch_size=8 by default
        self.patch_size = patch_size
        
        # CROMA features are all same dim (768 for Base)
        encoder_channels = [3, encoder_dim, encoder_dim, encoder_dim, encoder_dim]
        
        self.feature_proj = nn.ModuleDict({
            'block_0': nn.Conv2d(encoder_dim, encoder_dim, 1, bias=False),
            'block_2': nn.Conv2d(encoder_dim, encoder_dim, 1, bias=False),
            'block_4': nn.Conv2d(encoder_dim, encoder_dim, 1, bias=False),
            'norm':    nn.Conv2d(encoder_dim, encoder_dim, 1, bias=False),
        })
        
        self.decoder = UPerNetDecoder(
            encoder_channels=encoder_channels,
            encoder_depth=4,
            decoder_channels=decoder_channels,
            use_norm="batchnorm",
        )
        
        self.segmentation_head = SegmentationHead(
            in_channels=decoder_channels,
            out_channels=num_classes,
            activation=None,
            kernel_size=1,
            upsampling=4, 
        )
        self.dropout = nn.Dropout2d(dropout)
    
    def reshape_vit_features(self, features, H, W):
        # CROMA does NOT use a CLS token, so we do NOT skip the first token.
        # Direct reshape.
        B, N, D = features.shape
        return features.transpose(1, 2).reshape(B, D, H, W)
    
    def forward(self, features_dict, target_size, dummy_input=None):
        B, N, D = features_dict['norm'].shape
        # Calculate grid size based on N (Num patches)
        # N = (H/8 * W/8)
        H_feat = W_feat = int(np.sqrt(N))
        
        feat_1 = self.feature_proj['block_0'](self.reshape_vit_features(features_dict['block_0'], H_feat, W_feat))
        feat_3 = self.feature_proj['block_2'](self.reshape_vit_features(features_dict['block_2'], H_feat, W_feat))
        feat_5 = self.feature_proj['block_4'](self.reshape_vit_features(features_dict['block_4'], H_feat, W_feat))
        feat_norm = self.feature_proj['norm'](self.reshape_vit_features(features_dict['norm'], H_feat, W_feat))
        
        # Resize dummy input to match feature map size for UPerNet requirements. its a placeholder. upernet just uses the last 4 features.
        if dummy_input is None: 
            dummy_input = torch.zeros(B, 3, H_feat, H_feat, device=feat_1.device)
        else: 
            # Slice dummy input to 3 channels just in case, though UPerNet only uses shape
            dummy_input = F.interpolate(dummy_input, size=(H_feat, H_feat), mode='bilinear', align_corners=False)
        
        features_list = [dummy_input, feat_1, feat_3, feat_5, feat_norm]
        x = self.segmentation_head(self.dropout(self.decoder(features_list)))
        return F.interpolate(x, size=target_size, mode='bilinear', align_corners=False)

# ============================================================================
# 3. Trainer & Utils (KEPT MOSTLY SAME)
# ============================================================================

# Adjusted stats for 3-channel SAR (VV, VH, Avg)
S1_MEAN = [166.36, 88.45] 
S1_STD = [64.83, 43.07]

class GPUAugmentation(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        # 1. Geometric Augmentations (Applied to BOTH Image and Mask)
        self.aug = C.AugmentationSequential(
            K.RandomHorizontalFlip(p=0.5),
            K.RandomVerticalFlip(p=0.5),
            data_keys=["input", "mask"], # synchronized transform
            same_on_batch=False
        )
        
        # 2. Normalization (Applied ONLY to Image manually)
        self.normalize = K.Normalize(mean=torch.tensor(mean), std=torch.tensor(std))

    def forward(self, img, mask):
        # Apply geometry to both (flips happen in sync)
        img, mask = self.aug(img, mask)
        
        # Apply normalization ONLY to the image
        img = self.normalize(img)
        
        return img, mask

class SegmentationTrainer:
    def __init__(self, croma_model, decoder, train_loader, device='cuda', 
                 encoder_lr=1e-5, decoder_lr=3e-4, num_epochs=100, checkpoint_dir='checkpoints/CROMA+UPerNet'):
        self.croma_model = croma_model.to(device)
        self.decoder = decoder.to(device)
        self.train_loader = train_loader
        self.device = device
        self.num_epochs = num_epochs
        self.checkpoint_dir = checkpoint_dir
        
        # CHANGED: Initialize Croma Extractor
        self.extractor_hook = CromaFeatureExtractor(self.croma_model)
        self.augmentor = GPUAugmentation(mean=S1_MEAN, std=S1_STD).to(device)
        
        # Freeze CROMA logic if needed, or train all.
        # CROMA pretrained is usually robust. We can fine-tune.
        self.croma_model.train()
        for param in self.croma_model.parameters(): param.requires_grad = True 
        
        self.optimizer = AdamW([
            {'params': self.croma_model.parameters(), 'lr': encoder_lr},
            {'params': self.decoder.parameters(), 'lr': decoder_lr}
        ], weight_decay=0.01)
        
        self.scheduler = CosineAnnealingLR(self.optimizer, T_max=num_epochs)
        self.criterion = nn.CrossEntropyLoss()
        self.scaler = GradScaler()
        self.best_train_iou = 0.0
        self.history = []
        os.makedirs(checkpoint_dir, exist_ok=True)
    
    def train_epoch(self, epoch):
        self.decoder.train()
        self.croma_model.train()
        total_loss, total_iou = 0.0, 0.0
        
        pbar = tqdm(self.train_loader, desc=f'Epoch {epoch+1}/{self.num_epochs}')
        for batch in pbar:
            images = batch['image'].to(self.device, non_blocking=True)
            masks = batch['mask'].to(self.device, non_blocking=True).float()
            # if len(masks.shape) == 3:
            #     masks = masks.unsqueeze(1)
            
            images, masks = self.augmentor(images,masks)
            masks = masks.long().squeeze(1)
            images = images.contiguous()
            masks = masks.contiguous()

            with autocast('cuda'):
                # CHANGED: Extract Features using CROMA logic
                features_dict = extract_patch_features(self.croma_model, images, self.extractor_hook)
                
                # Pass dummy_input (3 channels) for UPerNet internals
                logits = self.decoder(features_dict, masks.shape[-2:], dummy_input=images)
                loss = self.criterion(logits, masks)
            
            self.optimizer.zero_grad()
            self.scaler.scale(loss).backward()
            self.scaler.step(self.optimizer)
            self.scaler.update()
            
            loss_item = loss.item()
            iou = self.compute_iou(logits.detach(), masks)
            total_loss += loss_item
            total_iou += iou
            
            pbar.set_postfix({'loss': f'{loss_item:.4f}', 'IoU': f'{iou:.4f}'})
        
        return {'loss': total_loss / len(self.train_loader), 'iou': total_iou / len(self.train_loader)}

    def compute_iou(self, logits, masks):
        preds = torch.argmax(logits, dim=1)
        num_classes = logits.shape[1]
        iou_per_class = []
        for cls in range(num_classes):
            pred_cls = (preds == cls)
            mask_cls = (masks == cls)
            intersection = (pred_cls & mask_cls).sum().float()
            union = (pred_cls | mask_cls).sum().float()
            if union > 0: iou_per_class.append((intersection / union).item())
        return np.mean(iou_per_class) if iou_per_class else 0.0
    
    def train(self):
        for epoch in range(self.num_epochs):
            metrics = self.train_epoch(epoch)
            self.scheduler.step()
            print(f"\nEpoch {epoch+1} - Loss: {metrics['loss']:.4f}, IoU: {metrics['iou']:.4f}")
            self.history.append({'epoch': epoch+1, **metrics})
            
            if metrics['iou'] > self.best_train_iou:
                self.best_train_iou = metrics['iou']
                self.save_checkpoint(epoch, metrics, is_best=True)
            
            if (epoch + 1) % 10 == 0:
                self.save_checkpoint(epoch, metrics, is_best=False)

    def save_checkpoint(self, epoch, metrics, is_best=False):
        checkpoint = {
            'epoch': epoch,
            'encoder_state_dict': self.croma_model.state_dict(),
            'decoder_state_dict': self.decoder.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'metrics': metrics,
        }
        torch.save(checkpoint, os.path.join(self.checkpoint_dir, f'ckpt_epoch_{epoch}.pth'))
        if is_best:
            torch.save(checkpoint, os.path.join(self.checkpoint_dir, 'best_model.pth'))

# ============================================================================
# 4. Dataset (UNCHANGED)
# ============================================================================

class SegmentationDataset(Dataset):
    def __init__(self, root_dir: str):
        self.root_dir = Path(root_dir)
        self.mask_paths = sorted(list(self.root_dir.glob("labels/*.png")))
        self.vv_dir = self.root_dir / "vv"
        self.vh_dir = self.root_dir / "vh"

    def __len__(self):
        return len(self.mask_paths)

    def __getitem__(self, idx):
        mask_path = self.mask_paths[idx]
        filename = mask_path.name
        
        with rasterio.open(self.vv_dir / filename) as f:
            vv = f.read().astype('float32')
        with rasterio.open(self.vh_dir / filename) as f:
            vh = f.read().astype('float32')
        with rasterio.open(mask_path) as f:
            label = f.read().astype('float32')
        
        label = np.where(label > 128, 1, 0)

        # Dataset still produces 3 channels (VV, VH, Avg).
        # We will slice this in the Model Wrapper for CROMA (which needs 2).
        s1_img = np.concatenate((vv, vh), axis=0)
        
        return {"image": torch.from_numpy(s1_img), "mask": torch.from_numpy(label).long()}

# ============================================================================
# 5. Main (MODIFIED for CROMA)
# ============================================================================

def main():
    config = {
        "encoder_dim": 768, "decoder_channels": 256, "num_classes": 2,
        "patch_size": 8, # CHANGED: CROMA uses patch size 8
        "image_size": 224, 
        "batch_size": 40,  # Reduced batch size because Patch 8 uses 4x more tokens than Patch 16
        "encoder_lr": 1e-5, "decoder_lr": 3e-4, "num_epochs": 100,
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "head_dropout": 0.1
    }
    
    # --- MODIFIED: Load CROMA Encoder ---
    print("Loading CROMA SAR Encoder...")
    
    # 1. Path to your CROMA Checkpoint
    checkpoint_path = r"/home/arm/Desktop/ARM/Codes/CROMA/CROMA_base.pt"
    
    # 2. Instantiate CROMA Model
    # Important: Set image_resolution to 224 so ALiBi bias is calculated correctly
    croma_model = PretrainedCROMA(
        pretrained_path=checkpoint_path, 
        size='base', 
        modality='SAR', 
        image_resolution=config['image_size']
    )
    
    print("CROMA Checkpoint Loaded successfully.")

    print("Preparing Data...")
    train_dataset = SegmentationDataset(root_dir=r'/home/arm/Documents/ARM/tiled_dataset')
    
    train_loader = DataLoader(
        train_dataset, batch_size=config['batch_size'], shuffle=True,
        num_workers=10, pin_memory=True, persistent_workers=True, prefetch_factor=4
    )

    print("Initializing CromaUPerNet Decoder...")
    decoder = CromaUPerNet(
        encoder_dim=config['encoder_dim'],
        decoder_channels=config['decoder_channels'],
        num_classes=config['num_classes'],
        dropout=config['head_dropout'],
        patch_size=config['patch_size']
    )

    print(f"Starting Training (Batch Size: {config['batch_size']})...")
    trainer = SegmentationTrainer(
        croma_model=croma_model,
        decoder=decoder,
        train_loader=train_loader,
        device=config['device'],
        encoder_lr=config['encoder_lr'],
        decoder_lr=config['decoder_lr'],
        num_epochs=config['num_epochs']
    )
    trainer.train()

if __name__ == "__main__":
    main()

/home/arm/Desktop/ARM/Codes/armvenv/lib/python3.10/site-packages/kornia/feature/lightglue.py:30: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)
/home/arm/Desktop/ARM/Codes/armvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading CROMA SAR Encoder...
Initializing SAR encoder
CROMA Checkpoint Loaded successfully.
Preparing Data...
Initializing CromaUPerNet Decoder...
Starting Training (Batch Size: 40)...


Epoch 1/100:   0%|          | 0/3685 [00:00<?, ?it/s]/home/arm/Desktop/ARM/Codes/armvenv/lib/python3.10/site-packages/rasterio/__init__.py:304: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/home/arm/Desktop/ARM/Codes/armvenv/lib/python3.10/site-packages/rasterio/__init__.py:304: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/home/arm/Desktop/ARM/Codes/armvenv/lib/python3.10/site-packages/rasterio/__init__.py:304: NotGeoreferencedWarning: Dataset has no geotransform, gcps, or rpcs. The identity matrix will be returned.
  dataset = DatasetReader(path, driver=driver, sharing=sharing, **kwargs)
/home/arm/Desktop/ARM/Codes/armvenv/lib/python3.10/site-packages/rasterio/__init__.py:304: NotGeoreferencedWarning: Dataset has no 


Epoch 1 - Loss: 0.1032, IoU: 0.9159


Epoch 2/100: 100%|██████████| 3685/3685 [13:37<00:00,  4.51it/s, loss=0.0451, IoU=0.9645]


...


Epoch 85 - Loss: 0.0246, IoU: 0.9785


Epoch 86/100: 100%|██████████| 3685/3685 [14:06<00:00,  4.36it/s, loss=0.0221, IoU=0.9813]



Epoch 86 - Loss: 0.0246, IoU: 0.9785


Epoch 87/100: 100%|██████████| 3685/3685 [15:00<00:00,  4.09it/s, loss=0.0395, IoU=0.9665]



Epoch 87 - Loss: 0.0246, IoU: 0.9785
